In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 8 Lab: Hierarchy, Parameters & Generate

## Course: Accelerated HDL for Digital System Design — Week 2, Session 8

---

## Objectives

By the end of this lab, you will:
- Build a parameterized modulo-N counter and test it at 3+ configurations
- Use `generate for` to create multi-channel debounce infrastructure
- Assemble the most complex hierarchical system of the course so far
- (Stretch) Build a generic LFSR with width-dependent tap selection via `generate if`

## Prerequisites

- Watched Day 8 pre-class video (~45 min): hierarchy, parameters, generate
- All modules from Days 2–7 (provided in `starter/` for convenience)

## Deliverables

| # | Exercise | Time | What to Submit |
|---|----------|------|----------------|
| 1 | Parameterized Counter | 25 min | `counter_mod_n.v` + `tb_counter_mod_n.v` — 3 configs pass |
| 2 | Generate Multi-Debounce | 25 min | `go_board_input.v` + testbench |
| 3 | Hierarchical System | 30 min | `top_lab_instrument.v` — all buttons work, both displays |
| 4 | Generic LFSR (stretch) | 20 min | `lfsr_generic.v` — maximal-length at 3 widths |

**Primary deliverable:** Hierarchical design with 3+ levels, parameterized, running on Go Board.

---

## Exercise 1: Parameterized Counter Module (25 min)

**SLOs: 8.2, 8.3, 8.5**

Create `counter_mod_n.v` — a universal modulo-N counter with `$clog2` auto-sizing.

Use the starter file. Test at N=10, N=16, and N=60 in a single testbench.

**Required tests per configuration:**
1. Reset → count is 0
2. Count to max, verify wrap signal asserts at N-1
3. Count rolls over to 0 after wrap
4. Enable test: disable for 5 cycles, verify count holds

In [ ]:
!make sim TB=tb_counter_mod_n.v SRCS="counter_mod_n.v"

---

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile counter_mod_n.v
// =============================================================================
// counter_mod_n.v — Parameterized Modulo-N Counter (Starter)
// Day 8, Exercise 1
// =============================================================================

module counter_mod_n #(
    parameter N = 10
)(
    input  wire                      i_clk,
    input  wire                      i_reset,
    input  wire                      i_enable,
    output reg  [$clog2(N)-1:0]      o_count,
    output wire                      o_wrap
);

    // ---- TODO: Implement counter logic ----
    // On reset: o_count <= 0
    // On enable:
    //   If o_count == N-1: o_count <= 0  (wrap)
    //   Else: o_count <= o_count + 1
    // When not enabled: hold value


    // ---- TODO: Wrap signal ----
    // assign o_wrap = (i_enable && o_count == N-1);

endmodule

In [ ]:
%%writefile tb_counter_mod_n.v
// =============================================================================
// tb_counter_mod_n.v — Parameterized Counter Testbench (Starter)
// Day 8, Exercise 1
// =============================================================================

`timescale 1ns / 1ps

module tb_counter_mod_n;

    reg clk, reset, enable;

    // Config 1: N=10
    wire [3:0] count_10;
    wire wrap_10;
    counter_mod_n #(.N(10)) uut_10 (
        .i_clk(clk), .i_reset(reset), .i_enable(enable),
        .o_count(count_10), .o_wrap(wrap_10)
    );

    // Config 2: N=16
    wire [3:0] count_16;
    wire wrap_16;
    counter_mod_n #(.N(16)) uut_16 (
        .i_clk(clk), .i_reset(reset), .i_enable(enable),
        .o_count(count_16), .o_wrap(wrap_16)
    );

    // Config 3: N=60
    wire [5:0] count_60;
    wire wrap_60;
    counter_mod_n #(.N(60)) uut_60 (
        .i_clk(clk), .i_reset(reset), .i_enable(enable),
        .o_count(count_60), .o_wrap(wrap_60)
    );

    initial clk = 0;
    always #20 clk = ~clk;

    integer test_count = 0, fail_count = 0;

    // ---- TODO: Implement test tasks ----
    // Task: verify a counter wraps at exactly N-1 and resets to 0

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_counter_mod_n);

        reset = 1; enable = 1;
        repeat (3) @(posedge clk);
        reset = 0;

        // ---- TODO: Test N=10 counter ----
        // Run 10 cycles, verify wrap at count==9, count returns to 0

        // ---- TODO: Test N=16 counter ----

        // ---- TODO: Test N=60 counter ----

        // ---- TODO: Enable test — disable for 5 cycles, verify hold ----

        $display("");
        $display("========================================");
        $display("Counter tests: %0d / %0d passed",
                 test_count - fail_count, test_count);
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = counter_mod_n
SRCS     = counter_mod_n.v
TB       = tb_counter_mod_n.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_counter_mod_n.v counter_mod_n.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 2: Generate-Based Multi-Debounce (25 min)

**SLO: 8.4**

Create `go_board_input.v` using `generate for` to stamp out N debounce + edge-detect channels.

Test with a top module that uses all 4 buttons for different counter operations.

---

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile debounce.v
// =============================================================================
// debounce.v — Counter-Based Button Debouncer (Solution)
// Day 5, Exercise 1
// =============================================================================

module debounce #(
    parameter CLKS_TO_STABLE = 250_000
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;
    reg r_sync_0, r_sync_1;

    always @(posedge i_clk) begin
        // 2-FF synchronizer
        r_sync_0 <= i_bouncy;
        r_sync_1 <= r_sync_0;

        // Debounce logic
        if (r_sync_1 != o_clean) begin
            r_count <= r_count + 1;
            if (r_count == CLKS_TO_STABLE - 1) begin
                o_clean <= r_sync_1;
                r_count <= 0;
            end
        end else begin
            r_count <= 0;
        end
    end

endmodule

In [ ]:
%%writefile go_board_input.v
// =============================================================================
// go_board_input.v — Generate-Based Multi-Channel Input Processor (Starter — Provided)
// Day 8, Exercise 2 — study this code, then use it in Exercise 3
// =============================================================================

module go_board_input #(
    parameter N_BUTTONS   = 4,
    parameter CLK_FREQ    = 25_000_000,
    parameter DEBOUNCE_MS = 10
)(
    input  wire                  i_clk,
    input  wire [N_BUTTONS-1:0]  i_buttons_n,  // active-low raw
    output wire [N_BUTTONS-1:0]  o_buttons,     // active-high debounced
    output wire [N_BUTTONS-1:0]  o_press_edge,
    output wire [N_BUTTONS-1:0]  o_release_edge
);

    localparam CLKS_TO_STABLE = (CLK_FREQ / 1000) * DEBOUNCE_MS;

    genvar g;
    generate
        for (g = 0; g < N_BUTTONS; g = g + 1) begin : gen_btn
            wire w_clean;
            debounce #(.CLKS_TO_STABLE(CLKS_TO_STABLE)) db (
                .i_clk(i_clk),
                .i_bouncy(i_buttons_n[g]),
                .o_clean(w_clean)
            );

            assign o_buttons[g] = ~w_clean;

            reg r_prev;
            always @(posedge i_clk) r_prev <= o_buttons[g];

            assign o_press_edge[g]   = o_buttons[g] & ~r_prev;
            assign o_release_edge[g] = ~o_buttons[g] & r_prev;
        end
    endgenerate

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = go_board_input
SRCS     = debounce.v go_board_input.v
TB       = 
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

## Exercise 3: Hierarchical System Integration (30 min)

**SLOs: 8.1, 8.6**

This is the **Week 2 capstone**. Build `top_lab_instrument.v` — a "digital lab instrument" integrating:

```
top_lab_instrument
├── go_board_input (4-channel debounce + edge detect)
│   └── debounce [×4] (via generate)
├── counter_mod_n #(.N(256)) (8-bit main counter)
├── hex_to_7seg (display 1 — lower nibble)
├── hex_to_7seg (display 2 — upper nibble)
├── lfsr_8bit (random number generator)
└── heartbeat LEDs
```

**Behavior:**
- Button 1: reset everything
- Button 2: increment counter
- Button 3: load LFSR value into counter
- Button 4: step LFSR (next random number)
- Display 1: lower 4 bits (hex)
- Display 2: upper 4 bits (hex)

---

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile counter_mod_n.v
// =============================================================================
// counter_mod_n.v — Parameterized Modulo-N Counter (Starter)
// Day 8, Exercise 1
// =============================================================================

module counter_mod_n #(
    parameter N = 10
)(
    input  wire                      i_clk,
    input  wire                      i_reset,
    input  wire                      i_enable,
    output reg  [$clog2(N)-1:0]      o_count,
    output wire                      o_wrap
);

    // ---- TODO: Implement counter logic ----
    // On reset: o_count <= 0
    // On enable:
    //   If o_count == N-1: o_count <= 0  (wrap)
    //   Else: o_count <= o_count + 1
    // When not enabled: hold value


    // ---- TODO: Wrap signal ----
    // assign o_wrap = (i_enable && o_count == N-1);

endmodule

In [ ]:
%%writefile debounce.v
// =============================================================================
// debounce.v — Counter-Based Button Debouncer (Solution)
// Day 5, Exercise 1
// =============================================================================

module debounce #(
    parameter CLKS_TO_STABLE = 250_000
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;
    reg r_sync_0, r_sync_1;

    always @(posedge i_clk) begin
        // 2-FF synchronizer
        r_sync_0 <= i_bouncy;
        r_sync_1 <= r_sync_0;

        // Debounce logic
        if (r_sync_1 != o_clean) begin
            r_count <= r_count + 1;
            if (r_count == CLKS_TO_STABLE - 1) begin
                o_clean <= r_sync_1;
                r_count <= 0;
            end
        end else begin
            r_count <= 0;
        end
    end

endmodule

In [ ]:
%%writefile go_board_input.v
// =============================================================================
// go_board_input.v — Generate-Based Multi-Channel Input Processor (Starter — Provided)
// Day 8, Exercise 2 — study this code, then use it in Exercise 3
// =============================================================================

module go_board_input #(
    parameter N_BUTTONS   = 4,
    parameter CLK_FREQ    = 25_000_000,
    parameter DEBOUNCE_MS = 10
)(
    input  wire                  i_clk,
    input  wire [N_BUTTONS-1:0]  i_buttons_n,  // active-low raw
    output wire [N_BUTTONS-1:0]  o_buttons,     // active-high debounced
    output wire [N_BUTTONS-1:0]  o_press_edge,
    output wire [N_BUTTONS-1:0]  o_release_edge
);

    localparam CLKS_TO_STABLE = (CLK_FREQ / 1000) * DEBOUNCE_MS;

    genvar g;
    generate
        for (g = 0; g < N_BUTTONS; g = g + 1) begin : gen_btn
            wire w_clean;
            debounce #(.CLKS_TO_STABLE(CLKS_TO_STABLE)) db (
                .i_clk(i_clk),
                .i_bouncy(i_buttons_n[g]),
                .o_clean(w_clean)
            );

            assign o_buttons[g] = ~w_clean;

            reg r_prev;
            always @(posedge i_clk) r_prev <= o_buttons[g];

            assign o_press_edge[g]   = o_buttons[g] & ~r_prev;
            assign o_release_edge[g] = ~o_buttons[g] & r_prev;
        end
    endgenerate

endmodule

In [ ]:
%%writefile hex_to_7seg.v
// hex_to_7seg.v — Provided module (from Day 2)
module hex_to_7seg (
    input  wire [3:0] i_hex,
    output reg  [6:0] o_seg   // {a,b,c,d,e,f,g} active low
);
    always @(*) begin
        case (i_hex)
            4'h0: o_seg = 7'b0000001;  4'h1: o_seg = 7'b1001111;
            4'h2: o_seg = 7'b0010010;  4'h3: o_seg = 7'b0000110;
            4'h4: o_seg = 7'b1001100;  4'h5: o_seg = 7'b0100100;
            4'h6: o_seg = 7'b0100000;  4'h7: o_seg = 7'b0001111;
            4'h8: o_seg = 7'b0000000;  4'h9: o_seg = 7'b0000100;
            4'hA: o_seg = 7'b0001000;  4'hB: o_seg = 7'b1100000;
            4'hC: o_seg = 7'b0110001;  4'hD: o_seg = 7'b1000010;
            4'hE: o_seg = 7'b0110000;  4'hF: o_seg = 7'b0111000;
        endcase
    end
endmodule

In [ ]:
%%writefile lfsr_8bit.v
// =============================================================================
// lfsr_8bit.v — 8-bit LFSR (Solution — same as starter, provided complete)
// =============================================================================

module lfsr_8bit (
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_enable,
    output reg  [7:0] o_lfsr,
    output wire       o_valid
);

    wire w_feedback = o_lfsr[7] ^ o_lfsr[5] ^ o_lfsr[4] ^ o_lfsr[3];

    always @(posedge i_clk) begin
        if (i_reset)
            o_lfsr <= 8'h01;
        else if (i_enable)
            o_lfsr <= {o_lfsr[6:0], w_feedback};
    end

    assign o_valid = |o_lfsr;

endmodule

In [ ]:
%%writefile top_lab_instrument.v
// =============================================================================
// top_lab_instrument.v — Week 2 Capstone: Digital Lab Instrument (Starter)
// Day 8, Exercise 3
// =============================================================================

module top_lab_instrument (
    input  wire i_clk,
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1, o_led2, o_led3, o_led4,
    output wire o_segment1_a, o_segment1_b, o_segment1_c,
    output wire o_segment1_d, o_segment1_e, o_segment1_f, o_segment1_g,
    output wire o_segment2_a, o_segment2_b, o_segment2_c,
    output wire o_segment2_d, o_segment2_e, o_segment2_f, o_segment2_g
);

    // --- Input Processing ---
    wire [3:0] w_buttons, w_press;

    go_board_input #(
        .N_BUTTONS(4),
        .CLK_FREQ(25_000_000),
        .DEBOUNCE_MS(10)
    ) inputs (
        .i_clk(i_clk),
        .i_buttons_n({i_switch1, i_switch2, i_switch3, i_switch4}),
        .o_buttons(w_buttons),
        .o_press_edge(w_press),
        .o_release_edge()        // unused
    );

    wire w_reset  = w_buttons[3];   // sw1 held = reset
    wire w_inc    = w_press[2];     // sw2 press = increment
    wire w_load   = w_press[1];     // sw3 press = load from LFSR
    wire w_step   = w_press[0];     // sw4 press = step LFSR

    // --- TODO: LFSR ---
    // Instantiate lfsr_8bit with reset and step enable

    // --- TODO: Main Counter (8-bit, loadable) ---
    // reg [7:0] r_counter;
    // always @(posedge i_clk) begin
    //   if (w_reset)     → clear to 0
    //   else if (w_load) → load LFSR value
    //   else if (w_inc)  → increment
    // end

    // --- TODO: Display 1 — lower nibble of r_counter ---
    // hex_to_7seg disp1 (...)

    // --- TODO: Display 2 — upper nibble of r_counter ---
    // hex_to_7seg disp2 (...)

    // --- TODO: LEDs — heartbeat at different rates ---
    // reg [24:0] r_hb;
    // always @(posedge i_clk) r_hb <= r_hb + 1;
    // assign o_led1 = ~r_hb[24]; ...

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = top_lab_instrument
SRCS     = counter_mod_n.v debounce.v go_board_input.v hex_to_7seg.v lfsr_8bit.v top_lab_instrument.v
TB       = 
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

## Exercise 4 (Stretch): Parameterized LFSR (20 min)

**SLOs: 8.2, 8.4, 8.5**

Create `lfsr_generic.v` with `generate if` for width-dependent tap selection. Verify maximal-length at WIDTH=4 (15 states), WIDTH=8 (255), WIDTH=16 (65535).

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile lfsr_generic.v
// =============================================================================
// lfsr_generic.v — Parameterized LFSR (Starter)
// Day 8, Exercise 4 (Stretch)
// =============================================================================

module lfsr_generic #(
    parameter WIDTH = 8,
    parameter SEED  = 1
)(
    input  wire              i_clk,
    input  wire              i_reset,
    input  wire              i_enable,
    output reg [WIDTH-1:0]   o_lfsr
);

    wire w_feedback;

    // ---- TODO: Use generate-if for width-dependent taps ----
    // generate
    //     if (WIDTH == 4)
    //         assign w_feedback = o_lfsr[3] ^ o_lfsr[2];
    //     else if (WIDTH == 8)
    //         assign w_feedback = o_lfsr[7] ^ o_lfsr[5] ^ o_lfsr[4] ^ o_lfsr[3];
    //     else if (WIDTH == 16)
    //         assign w_feedback = o_lfsr[15] ^ o_lfsr[14] ^ o_lfsr[12] ^ o_lfsr[3];
    //     else
    //         assign w_feedback = ^o_lfsr;  // XOR reduction (not maximal)
    // endgenerate

    always @(posedge i_clk) begin
        if (i_reset)
            o_lfsr <= SEED[WIDTH-1:0];
        else if (i_enable)
            o_lfsr <= {o_lfsr[WIDTH-2:0], w_feedback};
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = lfsr_generic
SRCS     = lfsr_generic.v
TB       = 
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean